In [ ]:
#Imports
from python_files.I_data_prep import load_train_data, feat_select
from python_files.II_data_cleaning import data_clean, create_separate_var, outliers, impute, standardise, combine, drift_artifact, binning_and_saving
from python_files.III_data_splitting import read_data, data_type_split, DUMMY, split
from python_files.IV_model_training import XGmodel, lr_wrapper, SKmodel
from python_files.V_model_test import XGaccuracy, XGperf, SKparams, save_columns
from python_files.VI_model_selection import get_exp_model_res, get_production_model, comp_models, reg_best_model
from python_files.VII_deployment import wait_for_deployment, staging
from mlflow.tracking.client import MlflowClient
import pandas as pd
import warnings
import datetime
import mlflow
import os

In [ ]:
#Pandas version warnings need to be ignored
warnings.filterwarnings('ignore')
pd.set_option('display.float_format',lambda x: "%.3f" % x)

In [ ]:
#Constants and paths
raw_data_path="artifacts/raw_data.csv"
gold_data_path="artifacts/train_gold_data.csv"
artifact_drift_path="artifacts/columns_drift.json"
outlier_summary_loc="artifacts/outlier_summary"
xgboost_model_path = "artifacts/lead_model_xgboost.json"

current_date=datetime.datetime.now().strftime("%Y_%B_%d")
artifact_path = "model"
model_name = "lead_model"
ex_name="TBA"+str(current_date)
mlflow.set_experiment(ex_name)

model_version = 0
model_version+=1
client = MlflowClient()

# Data prep

In [ ]:
os.makedirs("artifacts", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("mlruns", exist_ok=True)
os.makedirs("mlruns/.trash", exist_ok=True)

In [ ]:
df, min_date, max_date = load_train_data(raw_data_path)
df = feat_select(df)

# Data cleaning

In [ ]:
data = data_clean(df)
data, cat_vars, cont_vars = create_separate_var(data)

outliers(data, cont_vars, outlier_summary_loc)

In [ ]:
imp_cat_vars, imp_cont_vars = impute(cat_vars, cont_vars)
cont_vars=standardise(cont_vars)

In [ ]:
combine(cont_vars, cat_vars)
drift_artifact(data, artifact_drift_path)

In [ ]:
binning_and_saving(data, gold_data_path)
gold_data = read_data(gold_data_path)
data, cat_vars, other_vars = data_type_split(gold_data)

In [ ]:
data = DUMMY(cat_vars, data, other_vars)
X_train, X_test, y_train, y_test = split(data)

# Training two models

In [ ]:
model1 = XGmodel(X_train, y_train)
model2 = SKmodel(ex_name, X_train, y_train, X_test, y_test)

# Testing and comparing

In [ ]:
mod1_y_pred_te, mod1_y_pred_tr = XGaccuracy(model1, X_train, y_train, X_test, y_test)
XGperf(model1, y_train, y_test, mod1_y_pred_te, mod1_y_pred_tr, xgboost_model_path)

In [ ]:
__ = SKparams()
__ = save_columns()

# Best model and deployment

In [ ]:
__ = get_exp_model_res()
__ = get_production_model()
__ = comp_models()
__ = reg_best_model()

In [ ]:
__ = wait_for_deployment()
__ = staging()